# 02 Data Cleaning, Validation and Cancer Cohort Construction


## Purpose of this notebook

Notebook 01 confirmed that the public SPARCS inpatient file contains the key variables needed for this portfolio project, including demographics, admission details, length of stay, discharge disposition, CCSR diagnosis groups, severity/risk variables, payer type, total charges and total costs.

This notebook prepares the analysis-ready dataset by cleaning key variables, validating the dataset, and creating defensible cancer cohort flags.

## Important methodological correction

The public SPARCS file used here does **not** provide raw ICD-10 diagnosis fields. Therefore, this project should not claim that the cancer cohort was defined using raw ICD-10 codes. Instead, cancer-related admissions are identified using **CCSR diagnosis codes and descriptions**.

This notebook creates:

1. `is_primary_cancer_admission`: a strict malignant cancer cohort.
2. `is_broader_cancer_related_admission`: a broader cancer-related cohort that includes uncertain/remission/treatment-related categories.
3. `cancer_cohort_type`: a categorical label for primary cancer, broader cancer-related admissions, and non-cancer admissions.
4. `cancer_group`: a higher-level cancer grouping for descriptive analysis and modelling.

## Notebook outputs

This notebook saves cleaned and documented outputs for later analysis:

| Output | Location | Purpose |
|---|---|---|
| Cleaned full dataset with derived variables | `data/interim/sparcs_2024_cleaned_with_flags.csv` | General cleaned dataset |
| Cancer cohort dataset | `data/processed/sparcs_2024_cancer_cohort_clean.csv` | Main file for utilisation/cost analysis |
| Model input dataset | `data/processed/sparcs_2024_model_input.csv` | Later regression modelling |
| Validation summary | `outputs/tables/notebook_02_validation_summary.csv` | Audit trail for data quality |
| Missingness summary | `outputs/tables/notebook_02_missingness_summary.csv` | Documentation |
| Cancer cohort counts | `outputs/tables/notebook_02_cancer_cohort_counts.csv` | Cohort construction checks |
| Cancer CCSR classification map | `outputs/tables/notebook_02_cancer_ccsr_classification_map.csv` | Transparent cohort definition |

In [1]:
# Core imports
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)

In [2]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

# Detect project directory whether notebook is run from root folder or /notebooks
current_dir = Path.cwd()

if current_dir.name == "notebooks":
    project_dir = current_dir.parent
else:
    project_dir = current_dir

data_raw = project_dir / "data" / "raw"
data_processed = project_dir / "data" / "processed"
outputs_tables = project_dir / "outputs" / "tables"

for folder in [data_raw, data_processed, outputs_tables]:
    folder.mkdir(parents=True, exist_ok=True)

print("Current directory:", current_dir)
print("Project directory:", project_dir)
print("Raw data folder:", data_raw)
print("Processed data folder:", data_processed)
print("Tables output folder:", outputs_tables)


Current directory: /Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/notebooks
Project directory: /Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost
Raw data folder: /Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/data/raw
Processed data folder: /Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/data/processed
Tables output folder: /Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outputs/tables


## 1. Load raw SPARCS extract

This notebook now prefers the larger SPARCS extract over the original 5,000-row development sample.

Loading priority:

1. `sparcs_2024_extract_100000.csv`
2. `sparcs_2024_extract_50000.csv`
3. `sparcs_2024_sample_5000.csv`

The printed output from the next cell must show `sparcs_2024_extract_50000.csv` and `Raw rows: 50000` before the scaled analysis can be trusted.


In [3]:
# Prefer larger SPARCS extracts over the original 5,000-row development sample.

preferred_files = [
    data_raw / "sparcs_2024_extract_100000.csv",
    data_raw / "sparcs_2024_extract_50000.csv",
    data_raw / "sparcs_2024_sample_5000.csv"
]

raw_path = None

for path in preferred_files:
    if path.exists():
        raw_path = path
        break

if raw_path is None:
    possible_files = sorted(data_raw.glob("*sparcs*.csv"))
    if len(possible_files) == 0:
        raise FileNotFoundError(
            "No SPARCS CSV file found in data/raw. "
            "Place sparcs_2024_extract_50000.csv in data/raw before rerunning Notebook 02."
        )
    raw_path = possible_files[-1]

print("Loading:", raw_path)

df_raw = pd.read_csv(raw_path, low_memory=False)

raw_file_name = raw_path.name
raw_file_path = str(raw_path)

print("Raw rows:", df_raw.shape[0])
print("Raw columns:", df_raw.shape[1])

df_raw.head()


Loading: /Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/data/raw/sparcs_2024_extract_50000.csv
Raw rows: 50000
Raw columns: 33


,health_service_area,hospital_county,operating_certificate_number,permanent_facility_id,facility_name,age_group,zip_code,gender,race,ethnicity,length_of_stay,type_of_admission,patient_disposition,discharge_year,ccsr_diagnosis_code,ccsr_diagnosis_description,ccsr_procedure_code,ccsr_procedure_description,apr_drg_code,apr_drg_description,apr_mdc_code,apr_mdc_description,apr_severity_of_illness_code,apr_severity_of_illness,apr_risk_of_mortality,apr_medical_surgical,payment_typology_1,payment_typology_2,payment_typology_3,birth_weight,emergency_department_indicator,total_charges,total_costs
0,Hudson Valley,Westchester,5957001.0,1139.0,WESTCHESTER MEDICAL CENTER,0-17,OOS,F,White,Not Span/Hispanic,1,Emergency,Home or Self Care,2024,SYM002,FEVER,NaN,NaN,722,FEVER AND INFLAMMATORY CONDITIONS,18,"INFECTIOUS AND PARASITIC DISEASES, SYSTEMIC OR...",2,Moderate,Minor,Medical,Private Health Insurance,NaN,NaN,NaN,Y,46814.00,6772.07
1,New York City,Queens,7003001.0,1628.0,FLUSHING HOSPITAL MEDICAL CENTER,0-17,113,M,White,Spanish/Hispanic,2,Emergency,Home or Self Care,2024,SYM002,FEVER,NaN,NaN,722,FEVER AND INFLAMMATORY CONDITIONS,18,"INFECTIOUS AND PARASITIC DISEASES, SYSTEMIC OR...",2,Moderate,Moderate,Medical,Medicaid,NaN,NaN,NaN,Y,13490.00,15464.30
2,New York City,New York,7002054.0,1458.0,NEW YORK-PRESBYTERIAN HOSPITAL - NEW YORK WEIL...,70 or Older,100,M,White,Not Span/Hispanic,2,Emergency,Home or Self Care,2024,SYM002,FEVER,ADM012,CHEMOTHERAPY,722,FEVER AND INFLAMMATORY CONDITIONS,18,"INFECTIOUS AND PARASITIC DISEASES, SYSTEMIC OR...",2,Moderate,Moderate,Medical,Medicare,Private Health Insurance,NaN,NaN,Y,49503.16,9324.77
3,New York City,New York,7002054.0,1464.0,NEW YORK-PRESBYTERIAN HOSPITAL - COLUMBIA PRES...,0-17,100,F,Other Race,Not Span/Hispanic,1,Emergency,Home or Self Care,2024,SYM002,FEVER,CNS002,LUMBAR PUNCTURE,722,FEVER AND INFLAMMATORY CONDITIONS,18,"INFECTIOUS AND PARASITIC DISEASES, SYSTEMIC OR...",1,Minor,Minor,Medical,Private Health Insurance,NaN,NaN,2700,Y,27827.66,7304.27
4,New York City,New York,7002032.0,1466.0,MOUNT SINAI WEST,18-29,100,F,Other Race,Spanish/Hispanic,1,Emergency,Home or Self Care,2024,SYM002,FEVER,ADM021,"ADMINISTRATION OF THERAPEUTIC SUBSTANCES, NEC",722,FEVER AND INFLAMMATORY CONDITIONS,18,"INFECTIOUS AND PARASITIC DISEASES, SYSTEMIC OR...",2,Moderate,Minor,Medical,Medicare,NaN,NaN,NaN,Y,32798.29,7948.10


In [4]:
# Standardise column names lightly.
# The SPARCS API already gives mostly snake_case names, but this protects the pipeline if spaces or capitals appear later.

def clean_column_name(col):
    col = str(col).strip().lower()
    col = re.sub(r"[^a-z0-9]+", "_", col)
    col = re.sub(r"_+", "_", col).strip("_")
    return col

original_columns = df_raw.columns.tolist()
df = df_raw.copy()
df.columns = [clean_column_name(c) for c in df.columns]

column_mapping = pd.DataFrame({
    "original_column_name": original_columns,
    "clean_column_name": df.columns
})

column_mapping.to_csv(outputs_tables / "notebook_02_column_name_mapping.csv", index=False)

print("Column mapping saved.")
print("Cleaned columns:")
print(df.columns.tolist())

Column mapping saved.
Cleaned columns:
['health_service_area', 'hospital_county', 'operating_certificate_number', 'permanent_facility_id', 'facility_name', 'age_group', 'zip_code', 'gender', 'race', 'ethnicity', 'length_of_stay', 'type_of_admission', 'patient_disposition', 'discharge_year', 'ccsr_diagnosis_code', 'ccsr_diagnosis_description', 'ccsr_procedure_code', 'ccsr_procedure_description', 'apr_drg_code', 'apr_drg_description', 'apr_mdc_code', 'apr_mdc_description', 'apr_severity_of_illness_code', 'apr_severity_of_illness', 'apr_risk_of_mortality', 'apr_medical_surgical', 'payment_typology_1', 'payment_typology_2', 'payment_typology_3', 'birth_weight', 'emergency_department_indicator', 'total_charges', 'total_costs']


## 2. Confirm required variables

This step prevents a common portfolio mistake: writing analysis code before checking whether the necessary variables exist. If a key variable is absent, the notebook should fail early and clearly.

In [5]:
required_columns = [
    "age_group",
    "gender",
    "race",
    "ethnicity",
    "length_of_stay",
    "type_of_admission",
    "patient_disposition",
    "discharge_year",
    "ccsr_diagnosis_code",
    "ccsr_diagnosis_description",
    "apr_severity_of_illness",
    "apr_risk_of_mortality",
    "apr_medical_surgical",
    "payment_typology_1",
    "emergency_department_indicator",
    "total_charges",
    "total_costs",
]

optional_columns = [
    "health_service_area",
    "hospital_county",
    "facility_name",
    "permanent_facility_id",
    "zip_code",
    "ccsr_procedure_code",
    "ccsr_procedure_description",
    "apr_drg_code",
    "apr_drg_description",
    "apr_mdc_code",
    "apr_mdc_description",
    "payment_typology_2",
    "payment_typology_3",
    "birth_weight",
]

available_required = [c for c in required_columns if c in df.columns]
missing_required = [c for c in required_columns if c not in df.columns]
available_optional = [c for c in optional_columns if c in df.columns]
missing_optional = [c for c in optional_columns if c not in df.columns]

availability_table = pd.DataFrame({
    "variable": required_columns + optional_columns,
    "status": ["required"] * len(required_columns) + ["optional"] * len(optional_columns),
    "available": [c in df.columns for c in required_columns + optional_columns]
})

availability_table.to_csv(outputs_tables / "notebook_02_variable_availability_check.csv", index=False)

display(availability_table)

if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")
else:
    print("All required columns are available.")

if missing_optional:
    print("Optional columns not found:", missing_optional)

,variable,status,available
0,age_group,required,True
1,gender,required,True
2,race,required,True
3,ethnicity,required,True
4,length_of_stay,required,True
5,type_of_admission,required,True
6,patient_disposition,required,True
7,discharge_year,required,True
8,ccsr_diagnosis_code,required,True
9,ccsr_diagnosis_description,required,True


All required columns are available.


## 3. Initial data quality snapshot

Before cleaning, create a baseline missingness and data type table. This acts as an audit trail and can be used later in the portfolio report.

In [6]:
def missingness_table(dataframe):
    return (
        pd.DataFrame({
            "column_name": dataframe.columns,
            "dtype": [str(dataframe[col].dtype) for col in dataframe.columns],
            "missing_count": [int(dataframe[col].isna().sum()) for col in dataframe.columns],
            "missing_percent": [round(float(dataframe[col].isna().mean() * 100), 2) for col in dataframe.columns],
            "unique_values": [int(dataframe[col].nunique(dropna=True)) for col in dataframe.columns]
        })
        .sort_values(["missing_percent", "column_name"], ascending=[False, True])
        .reset_index(drop=True)
    )

raw_missingness = missingness_table(df)
raw_missingness.to_csv(outputs_tables / "notebook_02_raw_missingness_summary.csv", index=False)
raw_missingness.head(20)

,column_name,dtype,missing_count,missing_percent,unique_values
0,birth_weight,str,45390,90.78,51
1,payment_typology_3,str,44873,89.75,9
2,payment_typology_2,str,28384,56.77,9
3,ccsr_procedure_code,str,14206,28.41,304
4,ccsr_procedure_description,str,14206,28.41,304
5,zip_code,str,3137,6.27,50
6,health_service_area,str,144,0.29,8
7,hospital_county,str,144,0.29,57
8,operating_certificate_number,float64,144,0.29,156
9,permanent_facility_id,float64,144,0.29,203


## 4. Clean length of stay

`length_of_stay` is read as a string because some values can be top-coded, for example `120+`. The model cannot use this directly.

This notebook creates:

- `length_of_stay_clean`: numeric length of stay.
- `length_of_stay_topcoded`: flag for values containing `+`.
- `length_of_stay_invalid`: flag for missing, zero, negative or unparseable values.

For `120+`, the numeric value is set to `120`, with the top-coded flag preserved. This is defensible for descriptive and regression work as long as the limitation is documented.

In [7]:
def clean_length_of_stay(series):
    raw = series.astype(str).str.strip()
    topcoded = raw.str.contains(r"\+", na=False)
    numeric_part = raw.str.extract(r"(\d+)", expand=False)
    numeric = pd.to_numeric(numeric_part, errors="coerce")
    invalid = numeric.isna() | (numeric <= 0)
    return numeric, topcoded, invalid

df["length_of_stay_raw"] = df["length_of_stay"].astype(str).str.strip()
df["length_of_stay_clean"], df["length_of_stay_topcoded"], df["length_of_stay_invalid"] = clean_length_of_stay(df["length_of_stay"])

los_quality = pd.DataFrame({
    "metric": [
        "rows",
        "missing_or_unparseable_los",
        "non_positive_los",
        "topcoded_los",
        "min_los_clean",
        "median_los_clean",
        "max_los_clean"
    ],
    "value": [
        len(df),
        int(df["length_of_stay_clean"].isna().sum()),
        int((df["length_of_stay_clean"] <= 0).sum()),
        int(df["length_of_stay_topcoded"].sum()),
        float(df["length_of_stay_clean"].min(skipna=True)),
        float(df["length_of_stay_clean"].median(skipna=True)),
        float(df["length_of_stay_clean"].max(skipna=True))
    ]
})

los_quality.to_csv(outputs_tables / "notebook_02_los_quality_summary.csv", index=False)
los_quality

,metric,value
0,rows,50000.0
1,missing_or_unparseable_los,0.0
2,non_positive_los,0.0
3,topcoded_los,49.0
4,min_los_clean,1.0
5,median_los_clean,3.0
6,max_los_clean,120.0


In [8]:
# Inspect raw LOS values that were top-coded or invalid.
los_problem_examples = df.loc[
    df["length_of_stay_topcoded"] | df["length_of_stay_invalid"],
    ["length_of_stay", "length_of_stay_clean", "length_of_stay_topcoded", "length_of_stay_invalid"]
].head(20)

los_problem_examples

,length_of_stay,length_of_stay_clean,length_of_stay_topcoded,length_of_stay_invalid
523,120+,120,True,False
577,120+,120,True,False
782,120+,120,True,False
1522,120+,120,True,False
2868,120+,120,True,False
2869,120+,120,True,False
3415,120+,120,True,False
5260,120+,120,True,False
8471,120+,120,True,False
8472,120+,120,True,False


## 5. Clean costs and charges

SPARCS provides both `total_charges` and `total_costs`. For this portfolio, both are useful:

- `total_charges` reflects billed charges.
- `total_costs` is usually more useful for resource-use and policy-facing analysis.

This notebook creates cleaned numeric versions and log-transformed versions for later modelling.

In [9]:
def clean_money_column(series):
    cleaned = (
        series.astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
        .replace({"": np.nan, "nan": np.nan, "None": np.nan})
    )
    return pd.to_numeric(cleaned, errors="coerce")

for col in ["total_charges", "total_costs"]:
    df[f"{col}_clean"] = clean_money_column(df[col])
    df[f"{col}_missing"] = df[f"{col}_clean"].isna()
    df[f"{col}_nonpositive"] = df[f"{col}_clean"] <= 0
    df[f"log_{col}_clean"] = np.where(
        df[f"{col}_clean"] > 0,
        np.log(df[f"{col}_clean"]),
        np.nan
    )

cost_quality_rows = []
for col in ["total_charges", "total_costs"]:
    clean_col = f"{col}_clean"
    cost_quality_rows.extend([
        {"variable": col, "metric": "missing", "value": int(df[clean_col].isna().sum())},
        {"variable": col, "metric": "non_positive", "value": int((df[clean_col] <= 0).sum())},
        {"variable": col, "metric": "min", "value": float(df[clean_col].min(skipna=True))},
        {"variable": col, "metric": "median", "value": float(df[clean_col].median(skipna=True))},
        {"variable": col, "metric": "mean", "value": float(df[clean_col].mean(skipna=True))},
        {"variable": col, "metric": "max", "value": float(df[clean_col].max(skipna=True))},
    ])

cost_quality = pd.DataFrame(cost_quality_rows)
cost_quality.to_csv(outputs_tables / "notebook_02_cost_charge_quality_summary.csv", index=False)
cost_quality

,variable,metric,value
0,total_charges,missing,0.000000e+00
1,total_charges,non_positive,0.000000e+00
2,total_charges,min,3.481400e+02
3,total_charges,median,4.766041e+04
4,total_charges,mean,8.991079e+04
5,total_charges,max,1.109771e+07
6,total_costs,missing,0.000000e+00
7,total_costs,non_positive,0.000000e+00
8,total_costs,min,4.620000e+01
9,total_costs,median,1.417145e+04


## 6. Clean categorical variables

This step strips whitespace, standardises missing categories, and creates simplified variables for later analysis.

In [10]:
def clean_category(series, missing_label="Unknown"):
    cleaned = series.astype("string").str.strip()
    cleaned = cleaned.replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
    return cleaned.fillna(missing_label)

categorical_columns = [
    "age_group",
    "gender",
    "race",
    "ethnicity",
    "zip_code",
    "health_service_area",
    "hospital_county",
    "facility_name",
    "type_of_admission",
    "patient_disposition",
    "ccsr_diagnosis_code",
    "ccsr_diagnosis_description",
    "ccsr_procedure_code",
    "ccsr_procedure_description",
    "apr_drg_description",
    "apr_mdc_description",
    "apr_severity_of_illness",
    "apr_risk_of_mortality",
    "apr_medical_surgical",
    "payment_typology_1",
    "payment_typology_2",
    "payment_typology_3",
    "emergency_department_indicator",
]

for col in categorical_columns:
    if col in df.columns:
        df[col] = clean_category(df[col])

# Main simplified variables for later notebooks
if "payment_typology_1" in df.columns:
    df["primary_payer"] = df["payment_typology_1"]

if "emergency_department_indicator" in df.columns:
    df["emergency_department_flag"] = (
        df["emergency_department_indicator"]
        .str.upper()
        .map({"Y": 1, "YES": 1, "N": 0, "NO": 0})
    )

if "type_of_admission" in df.columns:
    df["is_emergency_admission"] = df["type_of_admission"].str.contains("emergency", case=False, na=False).astype(int)

if "patient_disposition" in df.columns:
    df["discharged_home_flag"] = df["patient_disposition"].str.contains("home|self care", case=False, na=False).astype(int)

# Save frequency tables for key categorical variables
freq_tables = []
for col in [
    "age_group", "gender", "race", "ethnicity", "type_of_admission", "patient_disposition",
    "apr_severity_of_illness", "apr_risk_of_mortality", "apr_medical_surgical",
    "primary_payer", "emergency_department_indicator"
]:
    if col in df.columns:
        temp = df[col].value_counts(dropna=False).reset_index()
        temp.columns = ["category", "count"]
        temp["variable"] = col
        temp["percent"] = (temp["count"] / len(df) * 100).round(2)
        freq_tables.append(temp[["variable", "category", "count", "percent"]])

categorical_frequency_summary = pd.concat(freq_tables, ignore_index=True) if freq_tables else pd.DataFrame()
categorical_frequency_summary.to_csv(outputs_tables / "notebook_02_categorical_frequency_summary.csv", index=False)

categorical_frequency_summary.head(30)

,variable,category,count,percent
0,age_group,70 or Older,14911,29.82
1,age_group,50-69,13412,26.82
2,age_group,30-49,10240,20.48
3,age_group,0-17,7053,14.11
4,age_group,18-29,4384,8.77
5,gender,F,27009,54.02
6,gender,M,22983,45.97
7,gender,U,8,0.02
8,race,White,25545,51.09
9,race,Other Race,14456,28.91


## 7. Define primary and broader cancer-related cohorts

The keyword search from Notebook 01 identified possible cancer-related CCSR categories. However, not all keyword matches should be treated as cancer admissions.

This notebook uses a stricter classification:

### Primary cancer cohort
Includes malignant cancers, leukaemia, lymphoma, myeloma and malignant neuroendocrine tumours.

### Broader cancer-related cohort
Includes the primary cancer cohort plus selected uncertain, remission or treatment-related neoplasm categories.

### Excluded from cancer cohorts
Benign neoplasms and nonmalignant breast conditions are excluded from the main cancer cohort because including them would overstate cancer-related admissions.

In [11]:
# Strict primary malignant cancer CCSR codes identified from the sample and retained for main analysis.
primary_cancer_codes = {
    "NEO002": "HEAD AND NECK CANCERS - LIP AND ORAL CAVITY",
    "NEO015": "GASTROINTESTINAL CANCERS - COLORECTAL",
    "NEO017": "GASTROINTESTINAL CANCERS - LIVER",
    "NEO021": "GASTROINTESTINAL CANCERS - ALL OTHER TYPES",
    "NEO022": "RESPIRATORY CANCERS",
    "NEO023": "BONE CANCER",
    "NEO030": "BREAST CANCER - ALL OTHER TYPES",
    "NEO032": "FEMALE REPRODUCTIVE SYSTEM CANCERS - CERVIX",
    "NEO033": "FEMALE REPRODUCTIVE SYSTEM CANCERS - OVARY",
    "NEO035": "FEMALE REPRODUCTIVE SYSTEM CANCERS - ENDOMETRIUM",
    "NEO039": "MALE REPRODUCTIVE SYSTEM CANCERS - PROSTATE",
    "NEO041": "MALE REPRODUCTIVE SYSTEM CANCERS - PENIS",
    "NEO043": "URINARY SYSTEM CANCERS - BLADDER",
    "NEO044": "URINARY SYSTEM CANCERS - URETER AND RENAL PELVIS",
    "NEO045": "URINARY SYSTEM CANCERS - KIDNEY",
    "NEO048": "NERVOUS SYSTEM CANCERS - BRAIN",
    "NEO050": "ENDOCRINE SYSTEM CANCERS - THYROID",
    "NEO051": "ENDOCRINE SYSTEM CANCERS - PANCREAS",
    "NEO057": "HODGKIN LYMPHOMA",
    "NEO058": "NON-HODGKIN LYMPHOMA",
    "NEO060": "LEUKEMIA - ACUTE MYELOID LEUKEMIA (AML)",
    "NEO062": "LEUKEMIA - CHRONIC MYELOID LEUKEMIA (CML)",
    "NEO064": "LEUKEMIA - ALL OTHER TYPES",
    "NEO065": "MULTIPLE MYELOMA",
    "NEO066": "MALIGNANT NEUROENDOCRINE TUMORS",
}

# Broader cancer-related categories for sensitivity or secondary cohort.
broader_cancer_related_codes = {
    "NEO072": "NEOPLASMS OF UNSPECIFIED NATURE OR UNCERTAIN BEHAVIOR",
    "NEO074": "CONDITIONS DUE TO NEOPLASM OR THE TREATMENT OF NEOPLASM",
    "NEO075": "LEUKEMIA IN REMISSION",
}

# Explicit exclusions from the cancer cohort.
excluded_cancer_keyword_codes = {
    "GEN017": "NONMALIGNANT BREAST CONDITIONS",
    "NEO073": "BENIGN NEOPLASMS",
}

all_broad_codes = set(primary_cancer_codes) | set(broader_cancer_related_codes)

# Standardise CCSR diagnosis code before cohort flag creation.
df["ccsr_diagnosis_code_clean"] = (
    df["ccsr_diagnosis_code"]
    .astype("string")
    .str.strip()
    .str.upper()
)

df["is_primary_cancer_admission"] = df["ccsr_diagnosis_code_clean"].isin(primary_cancer_codes).astype(int)
df["is_broader_cancer_related_admission"] = df["ccsr_diagnosis_code_clean"].isin(all_broad_codes).astype(int)
df["is_excluded_cancer_keyword_match"] = df["ccsr_diagnosis_code_clean"].isin(excluded_cancer_keyword_codes).astype(int)

df["cancer_cohort_type"] = np.select(
    [
        df["is_primary_cancer_admission"].eq(1),
        df["ccsr_diagnosis_code_clean"].isin(broader_cancer_related_codes),
        df["is_excluded_cancer_keyword_match"].eq(1),
    ],
    [
        "primary_cancer",
        "broader_cancer_related_only",
        "excluded_keyword_match_not_cancer",
    ],
    default="non_cancer"
)

# Preserve the CCSR diagnosis description as the specific cancer category where relevant.
df["cancer_category"] = np.where(
    df["is_broader_cancer_related_admission"].eq(1),
    df["ccsr_diagnosis_description"],
    "Non-cancer or excluded category"
)

print(df["cancer_cohort_type"].value_counts(dropna=False))


cancer_cohort_type
non_cancer                           48259
primary_cancer                        1218
excluded_keyword_match_not_cancer      364
broader_cancer_related_only            159
Name: count, dtype: int64


In [12]:
# Higher-level cancer groups for clearer descriptive analysis and later modelling.

def assign_cancer_group(code):
    code = str(code)
    if code in {"NEO002"}:
        return "head_neck"
    if code in {"NEO015", "NEO017", "NEO021"}:
        return "gastrointestinal"
    if code in {"NEO022"}:
        return "respiratory"
    if code in {"NEO023"}:
        return "bone"
    if code in {"NEO030"}:
        return "breast"
    if code in {"NEO032", "NEO033", "NEO035"}:
        return "female_reproductive"
    if code in {"NEO039", "NEO041"}:
        return "male_reproductive"
    if code in {"NEO043", "NEO044", "NEO045"}:
        return "urinary"
    if code in {"NEO048"}:
        return "nervous_system"
    if code in {"NEO050", "NEO051"}:
        return "endocrine"
    if code in {"NEO057", "NEO058"}:
        return "lymphoma"
    if code in {"NEO060", "NEO062", "NEO064", "NEO075"}:
        return "leukemia"
    if code in {"NEO065"}:
        return "multiple_myeloma"
    if code in {"NEO066"}:
        return "neuroendocrine"
    if code in {"NEO072", "NEO074"}:
        return "broader_cancer_related"
    if code in excluded_cancer_keyword_codes:
        return "excluded_keyword_match_not_cancer"
    return "non_cancer"

df["cancer_group"] = df["ccsr_diagnosis_code_clean"].apply(assign_cancer_group)

cancer_group_counts = (
    df.groupby(["cancer_cohort_type", "cancer_group", "ccsr_diagnosis_code", "ccsr_diagnosis_description"], dropna=False)
    .size()
    .reset_index(name="admissions")
    .sort_values(["cancer_cohort_type", "admissions"], ascending=[True, False])
)

cancer_group_counts.to_csv(outputs_tables / "notebook_02_cancer_group_counts.csv", index=False)
cancer_group_counts.head(50)

,cancer_cohort_type,cancer_group,ccsr_diagnosis_code,ccsr_diagnosis_description,admissions
1,broader_cancer_related_only,broader_cancer_related,NEO074,CONDITIONS DUE TO NEOPLASM OR THE TREATMENT OF...,104
0,broader_cancer_related_only,broader_cancer_related,NEO072,NEOPLASMS OF UNSPECIFIED NATURE OR UNCERTAIN B...,48
2,broader_cancer_related_only,leukemia,NEO075,LEUKEMIA IN REMISSION,7
4,excluded_keyword_match_not_cancer,excluded_keyword_match_not_cancer,NEO073,BENIGN NEOPLASMS,342
3,excluded_keyword_match_not_cancer,excluded_keyword_match_not_cancer,GEN017,NONMALIGNANT BREAST CONDITIONS,22
317,non_cancer,non_cancer,PNL001,LIVEBORN,4369
137,non_cancer,non_cancer,INF002,SEPTICEMIA,3120
31,non_cancer,non_cancer,CIR019,HEART FAILURE,1301
83,non_cancer,non_cancer,END003,DIABETES MELLITUS WITH COMPLICATION,973
223,non_cancer,non_cancer,MBD017,ALCOHOL-RELATED DISORDERS,947


In [13]:
# Save the cancer CCSR classification map for transparent reporting.
classification_rows = []

for code, desc in primary_cancer_codes.items():
    classification_rows.append({
        "ccsr_diagnosis_code": code,
        "ccsr_diagnosis_description": desc,
        "classification": "primary_cancer",
        "included_in_primary_cancer_cohort": 1,
        "included_in_broader_cancer_related_cohort": 1,
        "rationale": "Malignant cancer, haematological malignancy or malignant neuroendocrine tumour."
    })

for code, desc in broader_cancer_related_codes.items():
    classification_rows.append({
        "ccsr_diagnosis_code": code,
        "ccsr_diagnosis_description": desc,
        "classification": "broader_cancer_related_only",
        "included_in_primary_cancer_cohort": 0,
        "included_in_broader_cancer_related_cohort": 1,
        "rationale": "Cancer-related, remission, uncertain neoplasm or treatment-related category; retained only for broader sensitivity analysis."
    })

for code, desc in excluded_cancer_keyword_codes.items():
    classification_rows.append({
        "ccsr_diagnosis_code": code,
        "ccsr_diagnosis_description": desc,
        "classification": "excluded_keyword_match_not_cancer",
        "included_in_primary_cancer_cohort": 0,
        "included_in_broader_cancer_related_cohort": 0,
        "rationale": "Keyword match but not appropriate for primary cancer cohort."
    })

cancer_ccsr_classification_map = pd.DataFrame(classification_rows)
cancer_ccsr_classification_map["cancer_group"] = cancer_ccsr_classification_map["ccsr_diagnosis_code"].apply(assign_cancer_group)
cancer_ccsr_classification_map = cancer_ccsr_classification_map.sort_values("ccsr_diagnosis_code")

cancer_ccsr_classification_map.to_csv(outputs_tables / "notebook_02_cancer_ccsr_classification_map.csv", index=False)
cancer_ccsr_classification_map

,ccsr_diagnosis_code,ccsr_diagnosis_description,classification,included_in_primary_cancer_cohort,included_in_broader_cancer_related_cohort,rationale,cancer_group
28,GEN017,NONMALIGNANT BREAST CONDITIONS,excluded_keyword_match_not_cancer,0,0,Keyword match but not appropriate for primary ...,excluded_keyword_match_not_cancer
0,NEO002,HEAD AND NECK CANCERS - LIP AND ORAL CAVITY,primary_cancer,1,1,"Malignant cancer, haematological malignancy or...",head_neck
1,NEO015,GASTROINTESTINAL CANCERS - COLORECTAL,primary_cancer,1,1,"Malignant cancer, haematological malignancy or...",gastrointestinal
2,NEO017,GASTROINTESTINAL CANCERS - LIVER,primary_cancer,1,1,"Malignant cancer, haematological malignancy or...",gastrointestinal
3,NEO021,GASTROINTESTINAL CANCERS - ALL OTHER TYPES,primary_cancer,1,1,"Malignant cancer, haematological malignancy or...",gastrointestinal
4,NEO022,RESPIRATORY CANCERS,primary_cancer,1,1,"Malignant cancer, haematological malignancy or...",respiratory
5,NEO023,BONE CANCER,primary_cancer,1,1,"Malignant cancer, haematological malignancy or...",bone
6,NEO030,BREAST CANCER - ALL OTHER TYPES,primary_cancer,1,1,"Malignant cancer, haematological malignancy or...",breast
7,NEO032,FEMALE REPRODUCTIVE SYSTEM CANCERS - CERVIX,primary_cancer,1,1,"Malignant cancer, haematological malignancy or...",female_reproductive
8,NEO033,FEMALE REPRODUCTIVE SYSTEM CANCERS - OVARY,primary_cancer,1,1,"Malignant cancer, haematological malignancy or...",female_reproductive


## 8. Create analysis-ready dataset

The next notebooks will focus on cancer-related admissions. For the main analysis, use the **primary cancer cohort**. The broader cohort is retained for sensitivity analysis.

For later modelling, this notebook keeps key variables only and removes records that cannot support the main outcomes.

In [14]:
# Define analysis columns that will be carried forward.
analysis_columns = [
    # identifiers/context
    "discharge_year", "health_service_area", "hospital_county", "facility_name", "permanent_facility_id", "zip_code",

    # demographics
    "age_group", "gender", "race", "ethnicity",

    # admission and utilisation
    "type_of_admission", "patient_disposition", "emergency_department_indicator",
    "emergency_department_flag", "is_emergency_admission", "discharged_home_flag",
    "length_of_stay_raw", "length_of_stay_clean", "length_of_stay_topcoded", "length_of_stay_invalid",

    # diagnosis and cohort
    "ccsr_diagnosis_code", "ccsr_diagnosis_code_clean", "ccsr_diagnosis_description",
    "cancer_cohort_type", "is_primary_cancer_admission", "is_broader_cancer_related_admission",
    "is_excluded_cancer_keyword_match", "cancer_category", "cancer_group",

    # severity/risk
    "apr_drg_code", "apr_drg_description", "apr_mdc_code", "apr_mdc_description",
    "apr_severity_of_illness_code", "apr_severity_of_illness", "apr_risk_of_mortality", "apr_medical_surgical",

    # procedure optional
    "ccsr_procedure_code", "ccsr_procedure_description",

    # payer
    "payment_typology_1", "primary_payer", "payment_typology_2", "payment_typology_3",

    # cost/charge outcomes
    "total_charges", "total_charges_clean", "log_total_charges_clean",
    "total_costs", "total_costs_clean", "log_total_costs_clean",
]

analysis_columns = [c for c in analysis_columns if c in df.columns]

df_analysis_ready = df[analysis_columns].copy()

# Main cancer cohort: strict primary cancer admissions only.
df_primary_cancer = df_analysis_ready[df_analysis_ready["is_primary_cancer_admission"].eq(1)].copy()

# Broader cancer-related cohort: primary + broader cancer-related categories.
df_broader_cancer = df_analysis_ready[df_analysis_ready["is_broader_cancer_related_admission"].eq(1)].copy()

print("All cleaned rows:", df_analysis_ready.shape)
print("Primary cancer cohort rows:", df_primary_cancer.shape)
print("Broader cancer-related cohort rows:", df_broader_cancer.shape)

All cleaned rows: (50000, 49)
Primary cancer cohort rows: (1218, 49)
Broader cancer-related cohort rows: (1377, 49)


In [15]:
# Model input quality filter.
# Later models require valid LOS and positive costs/charges.

model_filter = (
    df_primary_cancer["length_of_stay_clean"].notna()
    & (df_primary_cancer["length_of_stay_clean"] > 0)
    & df_primary_cancer["total_costs_clean"].notna()
    & (df_primary_cancer["total_costs_clean"] > 0)
    & df_primary_cancer["total_charges_clean"].notna()
    & (df_primary_cancer["total_charges_clean"] > 0)
)

df_model_input = df_primary_cancer.loc[model_filter].copy()

model_input_exclusions = pd.DataFrame({
    "reason": [
        "primary_cancer_rows",
        "excluded_invalid_los_or_cost_charge",
        "final_model_input_rows"
    ],
    "rows": [
        len(df_primary_cancer),
        int((~model_filter).sum()),
        len(df_model_input)
    ]
})

model_input_exclusions.to_csv(outputs_tables / "notebook_02_model_input_exclusions.csv", index=False)
model_input_exclusions

,reason,rows
0,primary_cancer_rows,1218
1,excluded_invalid_los_or_cost_charge,0
2,final_model_input_rows,1218


## 9. Validation summaries

These tables document what changed during cleaning and cohort construction. They are not decorative; they are the audit trail that makes the portfolio look like serious healthcare data work rather than a loose Kaggle notebook.

In [16]:
cohort_counts = (
    df.groupby("cancer_cohort_type", dropna=False)
    .agg(
        admissions=("cancer_cohort_type", "size"),
        mean_los=("length_of_stay_clean", "mean"),
        median_los=("length_of_stay_clean", "median"),
        mean_total_charges=("total_charges_clean", "mean"),
        median_total_charges=("total_charges_clean", "median"),
        mean_total_costs=("total_costs_clean", "mean"),
        median_total_costs=("total_costs_clean", "median")
    )
    .reset_index()
)

cohort_counts["percent_of_all_admissions"] = (cohort_counts["admissions"] / len(df) * 100).round(2)
cohort_counts.to_csv(outputs_tables / "notebook_02_cancer_cohort_counts.csv", index=False)
cohort_counts

,cancer_cohort_type,admissions,mean_los,median_los,mean_total_charges,median_total_charges,mean_total_costs,median_total_costs,percent_of_all_admissions
0,broader_cancer_related_only,159,7.194969,4.0,128416.657170,77409.000,36487.944654,21382.580,0.32
1,excluded_keyword_match_not_cancer,364,3.651099,2.0,107858.125192,65101.230,28469.477637,19814.875,0.73
2,non_cancer,48259,5.706335,3.0,88252.822078,46535.450,25786.825237,13884.610,96.52
3,primary_cancer,1218,7.388342,4.0,145211.985608,91354.615,41849.427611,27048.660,2.44


In [17]:
# Cleaned missingness summary
cleaned_missingness = missingness_table(df_analysis_ready)
cleaned_missingness.to_csv(outputs_tables / "notebook_02_missingness_summary.csv", index=False)
cleaned_missingness.head(30)

,column_name,dtype,missing_count,missing_percent,unique_values
0,permanent_facility_id,float64,144,0.29,203
1,age_group,string,0,0.00,5
2,apr_drg_code,int64,0,0.00,332
3,apr_drg_description,string,0,0.00,332
4,apr_mdc_code,int64,0,0.00,26
5,apr_mdc_description,string,0,0.00,26
6,apr_medical_surgical,string,0,0.00,3
7,apr_risk_of_mortality,string,0,0.00,5
8,apr_severity_of_illness,string,0,0.00,5
9,apr_severity_of_illness_code,int64,0,0.00,5


In [18]:
validation_summary = pd.DataFrame({
    "metric": [
        "raw_file_name",
        "raw_file_path",
        "raw_rows",
        "raw_columns",
        "cleaned_analysis_rows",
        "cleaned_analysis_columns",
        "primary_cancer_rows",
        "broader_cancer_related_rows",
        "excluded_keyword_match_not_cancer_rows",
        "model_input_rows_primary_cancer",
        "topcoded_los_rows",
        "invalid_los_rows",
        "missing_or_invalid_total_charges_rows",
        "missing_or_invalid_total_costs_rows",
        "unique_primary_cancer_ccsr_codes_in_data",
        "unique_broader_cancer_related_ccsr_codes_in_data"
    ],
    "value": [
        raw_file_name,
        raw_file_path,
        int(df_raw.shape[0]),
        int(df_raw.shape[1]),
        int(df_analysis_ready.shape[0]),
        int(df_analysis_ready.shape[1]),
        int(df["is_primary_cancer_admission"].sum()),
        int(df["is_broader_cancer_related_admission"].sum()),
        int(df["is_excluded_cancer_keyword_match"].sum()),
        int(df_model_input.shape[0]),
        int(df["length_of_stay_topcoded"].sum()),
        int(df["length_of_stay_invalid"].sum()),
        int((df["total_charges_clean"].isna() | (df["total_charges_clean"] <= 0)).sum()),
        int((df["total_costs_clean"].isna() | (df["total_costs_clean"] <= 0)).sum()),
        int(df.loc[df["is_primary_cancer_admission"].eq(1), "ccsr_diagnosis_code_clean"].nunique()),
        int(df.loc[df["is_broader_cancer_related_admission"].eq(1), "ccsr_diagnosis_code_clean"].nunique()),
    ]
})

validation_summary.to_csv(outputs_tables / "notebook_02_validation_summary.csv", index=False)
validation_summary


,metric,value
0,raw_file_name,sparcs_2024_extract_50000.csv
1,raw_file_path,/Users/marissa/Desktop/portfolio/oncology-heal...
2,raw_rows,50000
3,raw_columns,33
4,cleaned_analysis_rows,50000
5,cleaned_analysis_columns,49
6,primary_cancer_rows,1218
7,broader_cancer_related_rows,1377
8,excluded_keyword_match_not_cancer_rows,364
9,model_input_rows_primary_cancer,1218


### Cohort size assessment

This check determines whether the current extract is large enough for final modelling. The minimum threshold is set at 500 primary cancer admissions.


In [19]:
minimum_primary_cancer_rows_for_final_modelling = 500

cohort_size_assessment = pd.DataFrame({
    "assessment_item": [
        "total_cleaned_rows",
        "primary_cancer_rows",
        "broader_cancer_related_rows",
        "model_input_rows",
        "minimum_primary_cancer_rows_for_final_modelling",
        "suitable_for_pipeline_testing",
        "suitable_for_final_modelling",
        "analysis_dataset_status",
        "recommended_next_step"
    ],
    "value": [
        int(df_analysis_ready.shape[0]),
        int(df["is_primary_cancer_admission"].sum()),
        int(df["is_broader_cancer_related_admission"].sum()),
        int(df_model_input.shape[0]),
        minimum_primary_cancer_rows_for_final_modelling,
        "Yes",
        "Yes" if len(df_model_input) >= minimum_primary_cancer_rows_for_final_modelling else "No",
        "final_analysis_ready" if len(df_model_input) >= minimum_primary_cancer_rows_for_final_modelling else "development_sample",
        (
            "Proceed to final descriptive analysis and modelling."
            if len(df_model_input) >= minimum_primary_cancer_rows_for_final_modelling
            else "Proceed to workflow testing only, then rerun Notebook 02 on a larger SPARCS extract before final regression modelling."
        )
    ]
})

cohort_size_assessment.to_csv(
    outputs_tables / "notebook_02_cohort_size_assessment.csv",
    index=False
)

display(cohort_size_assessment)


,assessment_item,value
0,total_cleaned_rows,50000
1,primary_cancer_rows,1218
2,broader_cancer_related_rows,1377
3,model_input_rows,1218
4,minimum_primary_cancer_rows_for_final_modelling,500
5,suitable_for_pipeline_testing,Yes
6,suitable_for_final_modelling,Yes
7,analysis_dataset_status,final_analysis_ready
8,recommended_next_step,Proceed to final descriptive analysis and mode...


## 10. Save cleaned datasets

The outputs below will be used in later notebooks:

- Notebook 03: cancer cohort construction and descriptive cohort profile.
- Notebook 04: utilisation analysis.
- Notebook 05: cost and length-of-stay modelling.
- Notebook 06: export final tables and figures.

In [20]:
# Save CSV outputs to data/processed.

data_processed.mkdir(parents=True, exist_ok=True)

cleaned_full_path = data_processed / "sparcs_2024_cleaned_with_flags.csv"
primary_cancer_path = data_processed / "sparcs_2024_primary_cancer_cohort_clean.csv"
broader_cancer_path = data_processed / "sparcs_2024_broader_cancer_related_cohort_clean.csv"
model_input_path = data_processed / "sparcs_2024_model_input.csv"

df_analysis_ready.to_csv(cleaned_full_path, index=False)
df_primary_cancer.to_csv(primary_cancer_path, index=False)
df_broader_cancer.to_csv(broader_cancer_path, index=False)
df_model_input.to_csv(model_input_path, index=False)

print("Saved cleaned full dataset:", cleaned_full_path)
print("Saved primary cancer cohort:", primary_cancer_path)
print("Saved broader cancer-related cohort:", broader_cancer_path)
print("Saved model input dataset:", model_input_path)


Saved cleaned full dataset: /Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/data/processed/sparcs_2024_cleaned_with_flags.csv
Saved primary cancer cohort: /Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/data/processed/sparcs_2024_primary_cancer_cohort_clean.csv
Saved broader cancer-related cohort: /Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/data/processed/sparcs_2024_broader_cancer_related_cohort_clean.csv
Saved model input dataset: /Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/data/processed/sparcs_2024_model_input.csv


In [21]:
# Optional Parquet outputs for faster loading later.
# If pyarrow is not installed, this cell will skip Parquet export without failing the notebook.

try:
    df_analysis_ready.to_parquet(data_processed / "sparcs_2024_cleaned_with_flags.parquet", index=False)
    df_primary_cancer.to_parquet(data_processed / "sparcs_2024_primary_cancer_cohort_clean.parquet", index=False)
    df_broader_cancer.to_parquet(data_processed / "sparcs_2024_broader_cancer_related_cohort_clean.parquet", index=False)
    df_model_input.to_parquet(data_processed / "sparcs_2024_model_input.parquet", index=False)
    print("Parquet files saved successfully.")
except Exception as e:
    print("Parquet export skipped:", e)


Parquet files saved successfully.


## 11. Notebook conclusion

Notebook 02 created a cleaned, documented and analysis-ready dataset. The key methodological decision was to construct the cancer cohort using CCSR diagnosis categories rather than raw ICD-10 diagnosis codes, because the public SPARCS file does not expose raw ICD-10 diagnosis fields.

### Files to carry forward

Use these files in the next notebooks:

- Cleaned full dataset: `data/processed/sparcs_2024_cleaned_with_flags.csv`
- Main descriptive cancer cohort: `data/processed/sparcs_2024_primary_cancer_cohort_clean.csv`
- Broader sensitivity cohort: `data/processed/sparcs_2024_broader_cancer_related_cohort_clean.csv`
- Regression model input: `data/processed/sparcs_2024_model_input.csv`

### Validation files to check

Before moving forward, check:

- `outputs/tables/notebook_02_validation_summary.csv`
- `outputs/tables/notebook_02_cohort_size_assessment.csv`

The validation summary should show `raw_file_name = sparcs_2024_extract_50000.csv` and `raw_rows = 50000` if the scaled extract was loaded correctly.

### Next notebook

If `analysis_dataset_status = final_analysis_ready`, rerun Notebook 03, then Notebook 04, then Notebook 05 using the regenerated processed files.
